# DPA - Volume (Part 1): Network Traffic Analysis

Our research questions:
1. What traffic patterns distinguish normal activity from attacks?
2. Which network services and traffic characteristics are associated with different attack categories?

This notebook currently contains:
- Optional download of the two source datasets.
- Step 1: inspect the actual files and small traffic samples.
- Step 2: define explicit schemas and check column names.
- Step 3: test small-sample CSV ingestion and Parquet write/read.

Run the cells from top to bottom. Leave `DOWNLOAD_DATA = False` when the datasets are already available. Step 1 uses ordinary Python. Step 2 requires PySpark installed in the current kernel. Step 3 starts Spark and also requires Java; the configured Docker environment provides both.


## Optional — Download the source datasets

Leave `DOWNLOAD_DATA = False` to use files already on disk. Set it to `True` and run the next cell to download missing datasets and extract the traffic ZIP automatically. Existing source CSVs are preserved. This step uses ordinary Python in your current Jupyter kernel and does not require Docker or PySpark.

On a fresh download, allow at least 50 GiB of free space. Messages indicate downloading and extracting. Rerun the cell after an interruption; incomplete downloads restart from the beginning, while completed CSVs are preserved. Inputs are saved under the project's `data/` folder. Stop after this cell if you only want to obtain the datasets.

In [1]:
DOWNLOAD_DATA = False  # Optional: change to True to download missing source files.

from pathlib import Path

# Locate the project from notebooks/, the project root, or the outer workspace.
current_folder = Path.cwd().resolve()
root_candidates = [current_folder, *current_folder.parents]
ROOT = next((project for folder in root_candidates
             for project in (folder, folder / "DPA-Volume-Part1-Project")
             if (project / "downloaders" / "download_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the project folder or its notebooks/ folder.")
print("Project folder:", ROOT)

if DOWNLOAD_DATA:
    import runpy
    # Load reusable functions without invoking command-line argument parsing.
    downloader = runpy.run_path(str(ROOT / "downloaders" / "download_data.py"))
    downloader["prepare_iana"](ROOT / "data")
    downloader["prepare_traffic"](ROOT / "data")
    print("Source datasets are ready. No Spark processing was performed.")
else:
    print("Download skipped. Set DOWNLOAD_DATA = True and rerun this cell if needed.")

Project folder: /home/jovyan/work
Download skipped. Set DOWNLOAD_DATA = True and rerun this cell if needed.


## Step 1 - Inspect the actual source files (no Spark required)

**Purpose:** understand the raw columns and values before deciding their Spark types or cleaning rules. Run the optional download cell above first, with `DOWNLOAD_DATA = False` when files are already present.

We check every traffic header and read just **100 records at each of five positions per file** (start, 25%, 50%, 75%, 99%). This helps include later attack records without scanning gigabytes. These are systematic inspection windows, **not a random or representative statistical sample**. Do not use their counts to estimate attack frequencies. The traffic format is expected to have one CSV record per physical line; unexpected record widths cause an error for review.

The IANA CSV is small, so we inspect its complete registry to find port ranges and repeated keys. All fields remain raw strings. No records are cleaned, relabelled, or removed.

In [2]:
import csv
import io
import json
from collections import Counter
from datetime import datetime
from itertools import islice

traffic_folder = ROOT / "data" / "CSECICIDS2018_improved"
traffic_files = sorted(traffic_folder.glob("*.csv"))
iana_folder = ROOT / "data" / "Service and port name"
iana_files = sorted(iana_folder.glob("service-names-port-numbers*.csv"))
assert traffic_files, f"No traffic CSVs found in {traffic_folder}"
assert iana_files, f"No IANA registry found in {iana_folder}"
# Prefer the standard filename, while accepting the existing '(3)' filename.
registry_file = next((p for p in iana_files if p.name == "service-names-port-numbers.csv"), iana_files[0])

print("Traffic CSVs:", len(traffic_files))
for path in traffic_files:
    print(f"  {path.name}: {path.stat().st_size / 1024**3:.2f} GiB")
print("IANA:", registry_file.name)

Traffic CSVs: 10
  Friday-02-03-2018.csv: 3.44 GiB
  Friday-16-02-2018.csv: 3.92 GiB
  Friday-23-02-2018.csv: 3.18 GiB
  Thursday-01-03-2018.csv: 3.55 GiB
  Thursday-15-02-2018.csv: 2.79 GiB
  Thursday-22-02-2018.csv: 3.24 GiB
  Tuesday-20-02-2018.csv: 3.20 GiB
  Wednesday-14-02-2018.csv: 3.04 GiB
  Wednesday-21-02-2018.csv: 3.68 GiB
  Wednesday-28-02-2018.csv: 3.55 GiB
IANA: service-names-port-numbers (3).csv


### 1A - Compare the headers

`csv.reader` reads each header without loading the whole file. We compare column names **and their order**, and check for duplicate names. Exact names matter: `Total Fwd Packet` and `Total Bwd packets`, for example, need explicit treatment when we define the schema.

In [3]:
traffic_headers = {}
header_summary = []
for path in traffic_files:
    with path.open(encoding="utf-8-sig", newline="") as source:
        header = next(csv.reader(source))
    assert len(header) == len(set(header)), f"Duplicate column names in {path.name}"
    traffic_headers[path.name] = header
    reference_header = traffic_headers[traffic_files[0].name]
    header_summary.append({
        "file": path.name,
        "size_bytes": path.stat().st_size,
        "column_count": len(header),
        "matches_first_header": header == reference_header,
    })
    print(f"{path.name}: {len(header)} columns; same names/order = {header == reference_header}")

print("\nColumns in the first traffic file:")
for position, name in enumerate(reference_header, start=1):
    print(f"{position:2}. {name}")

Friday-02-03-2018.csv: 91 columns; same names/order = True
Friday-16-02-2018.csv: 91 columns; same names/order = True
Friday-23-02-2018.csv: 91 columns; same names/order = True
Thursday-01-03-2018.csv: 91 columns; same names/order = True
Thursday-15-02-2018.csv: 91 columns; same names/order = True
Thursday-22-02-2018.csv: 91 columns; same names/order = True
Tuesday-20-02-2018.csv: 91 columns; same names/order = True
Wednesday-14-02-2018.csv: 91 columns; same names/order = True
Wednesday-21-02-2018.csv: 91 columns; same names/order = True
Wednesday-28-02-2018.csv: 91 columns; same names/order = True

Columns in the first traffic file:
 1. id
 2. Flow ID
 3. Src IP
 4. Src Port
 5. Dst IP
 6. Dst Port
 7. Protocol
 8. Timestamp
 9. Flow Duration
10. Total Fwd Packet
11. Total Bwd packets
12. Total Length of Fwd Packet
13. Total Length of Bwd Packet
14. Fwd Packet Length Max
15. Fwd Packet Length Min
16. Fwd Packet Length Mean
17. Fwd Packet Length Std
18. Bwd Packet Length Max
19. Bwd Pa

### 1B - Inspect small windows of real traffic

The first 1,000 rows of each file were checked initially and were all benign. Reading only the beginning would miss attacks.

`seek` jumps to a byte position without reading the intervening data. We skip the current physical line (the header at position zero, or a partial record elsewhere), then let `csv.reader` parse at most 100 records. Every sampled record must match its file's header width. The original files are opened for reading only.

In [4]:
ROWS_PER_WINDOW = 100
WINDOW_POSITIONS = (0.0, 0.25, 0.5, 0.75, 0.99)

def inspect_window(path, header, fraction):
    with path.open("rb") as source:
        source.seek(int(path.stat().st_size * fraction))
        source.readline()  # Skip the header, or the partial line at this position.
        with io.TextIOWrapper(source, encoding="utf-8", newline="") as text:
            records = list(islice(csv.reader(text), ROWS_PER_WINDOW))
    assert all(len(row) == len(header) for row in records), f"Unexpected CSV record width in {path.name}"
    return [dict(zip(header, row)) for row in records]

sample_rows = []
window_summary = []
for path in traffic_files:
    for fraction in WINDOW_POSITIONS:
        rows = inspect_window(path, traffic_headers[path.name], fraction)
        sample_rows.extend(rows)
        window_summary.append({"file": path.name, "position_fraction": fraction, "sampled_rows": len(rows)})
    print(f"{path.name}: inspected {sum(w['sampled_rows'] for w in window_summary if w['file'] == path.name)} records")
print("Total inspected traffic records:", len(sample_rows))

Friday-02-03-2018.csv: inspected 500 records
Friday-16-02-2018.csv: inspected 500 records


Friday-23-02-2018.csv: inspected 500 records


Thursday-01-03-2018.csv: inspected 500 records
Thursday-15-02-2018.csv: inspected 500 records


Thursday-22-02-2018.csv: inspected 500 records
Tuesday-20-02-2018.csv: inspected 500 records
Wednesday-14-02-2018.csv: inspected 500 records


Wednesday-21-02-2018.csv: inspected 500 records


Wednesday-28-02-2018.csv: inspected 500 records
Total inspected traffic records: 5000


### 1C - Examine labels, protocols, timestamps, and measurements

These counts describe **only the inspected windows**. We keep attempted-attack labels as supplied; their treatment will require an explicit decision later. Numeric protocol codes must be reconciled with IANA's text protocols before joining.

We try the timestamp format `year-month-day hour:minute:second.fraction` and the same format without fractional seconds. The source strings are preserved. A timestamp without an offset does not establish its timezone. Units and negative sentinel values must be checked against dataset documentation before cleaning; a negative ICMP or window value should not automatically trigger row removal.

In [5]:
sample_labels = Counter(row["Label"] for row in sample_rows)
sample_protocols = Counter(row["Protocol"] for row in sample_rows)
sample_attempted = Counter(row["Attempted Category"] for row in sample_rows)
print("Labels in inspected windows:", dict(sample_labels))
print("Raw protocol codes:", dict(sample_protocols))
print("Attempted Category values:", dict(sample_attempted))

formats = ("%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S")
format_counts = Counter()
parsed_times = []
unparsed_times = []
for row in sample_rows:
    for timestamp_format in formats:
        try:
            parsed_times.append(datetime.strptime(row["Timestamp"], timestamp_format))
            format_counts[timestamp_format] += 1
            break
        except ValueError:
            continue
    else:
        unparsed_times.append(row["Timestamp"])
print("Timestamp formats matched:", dict(format_counts))
print("Unparsed sampled timestamps:", len(unparsed_times))
if parsed_times:
    print("Sample timestamp range:", min(parsed_times), "to", max(parsed_times))

# Display one raw example per observed label, without reading more records.
example_columns = ["Src IP", "Dst IP", "Dst Port", "Protocol", "Timestamp", "Flow Duration",
                   "Total Fwd Packet", "Total Bwd packets", "Label", "Attempted Category"]
examples = {}
for row in sample_rows:
    if row["Label"] not in examples:
        examples[row["Label"]] = {column: row[column] for column in example_columns}
print("\nOne raw example per observed label:")
print(json.dumps(examples, indent=2))

sentinel_examples = {}
for column in ("ICMP Code", "ICMP Type", "FWD Init Win Bytes", "Bwd Init Win Bytes"):
    sentinel_examples[column] = sorted({row[column] for row in sample_rows if row[column].startswith("-")})
print("Observed negative values requiring interpretation:", sentinel_examples)

Labels in inspected windows: {'BENIGN': 4500, 'DoS Hulk': 200, 'DDoS-LOIC-HTTP': 99, 'DDoS-LOIC-UDP': 1, 'SSH-BruteForce': 100, 'DDoS-HOIC': 100}
Raw protocol codes: {'6': 3330, '17': 1661, '0': 6, '1': 3}
Attempted Category values: {'-1': 5000}
Timestamp formats matched: {'%Y-%m-%d %H:%M:%S.%f': 5000}
Unparsed sampled timestamps: 0
Sample timestamp range: 2018-02-14 12:29:09.861592 to 2018-03-02 21:27:31.377018

One raw example per observed label:
{
  "BENIGN": {
    "Src IP": "172.31.66.5",
    "Dst IP": "72.21.91.29",
    "Dst Port": "80",
    "Protocol": "6",
    "Timestamp": "2018-03-02 12:47:05.189575",
    "Flow Duration": "116659398",
    "Total Fwd Packet": "17",
    "Total Bwd packets": "16",
    "Label": "BENIGN",
    "Attempted Category": "-1"
  },
  "DoS Hulk": {
    "Src IP": "18.219.193.20",
    "Dst IP": "172.31.69.25",
    "Dst Port": "80",
    "Protocol": "6",
    "Timestamp": "2018-02-16 17:45:58.594883",
    "Flow Duration": "231372",
    "Total Fwd Packet": "5",
  

### 1D - Inspect the IANA registry before choosing the join

This secondary dataset is small enough to inspect with Python's CSV reader. Some port values represent ranges, and some rows have no service or transport protocol. Multiple entries can share a literal `(Port Number, Transport Protocol)` key.

We count these situations here; we do not expand ranges, drop rows, choose a service, or join yet. A registered service is an association, not proof of the actual application that produced a flow.

In [6]:
with registry_file.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    registry_header = reader.fieldnames
    registry_rows = list(reader)  # Only the small IANA registry, never the traffic files.
print("IANA columns:", registry_header)
assert len(registry_header) == len(set(registry_header)), "Duplicate IANA column names"
assert {"Port Number", "Transport Protocol", "Service Name"}.issubset(registry_header)
assert all(None not in row and None not in row.values() for row in registry_rows), "Unexpected IANA record width"

registry_keys = Counter((row["Port Number"], row["Transport Protocol"]) for row in registry_rows)
range_ports = [row["Port Number"] for row in registry_rows if "-" in row["Port Number"]]
iana_summary = {
    "file": registry_file.name,
    "row_count": len(registry_rows),
    "column_count": len(registry_header),
    "transport_protocol_counts": dict(Counter(row["Transport Protocol"] for row in registry_rows)),
    "rows_with_port_ranges": len(range_ports),
    "port_range_examples": list(dict.fromkeys(range_ports))[:5],
    "rows_with_blank_port": sum(not row["Port Number"].strip() for row in registry_rows),
    "rows_with_blank_protocol": sum(not row["Transport Protocol"].strip() for row in registry_rows),
    "rows_with_blank_service": sum(not row["Service Name"].strip() for row in registry_rows),
    "repeated_literal_key_count": sum(count > 1 for count in registry_keys.values()),
    "extra_rows_on_repeated_literal_keys": sum(count - 1 for count in registry_keys.values()),
}
print(json.dumps(iana_summary, indent=2))
print("Example registered service:", next(row for row in registry_rows if row["Service Name"] and row["Port Number"].isdigit()))

IANA columns: ['Service Name', 'Port Number', 'Transport Protocol', 'Description', 'Assignee', 'Contact', 'Registration Date', 'Modification Date', 'Reference', 'Service Code', 'Unauthorized Use Reported', 'Assignment Notes']
{
  "file": "service-names-port-numbers (3).csv",
  "row_count": 14536,
  "column_count": 12,
  "transport_protocol_counts": {
    "tcp": 6608,
    "udp": 6357,
    "sctp": 93,
    "dccp": 11,
    "": 1467
  },
  "rows_with_port_ranges": 766,
  "port_range_examples": [
    "225-241",
    "249-255",
    "272-279",
    "288-299",
    "301-307"
  ],
  "rows_with_blank_port": 1073,
  "rows_with_blank_protocol": 1467,
  "rows_with_blank_service": 1724,
  "repeated_literal_key_count": 230,
  "extra_rows_on_repeated_literal_keys": 1300
}
Example registered service: {'Service Name': 'tcpmux', 'Port Number': '1', 'Transport Protocol': 'tcp', 'Description': 'TCP Port Service Multiplexer', 'Assignee': '[Mark_Lottor]', 'Contact': '[Mark_Lottor]', 'Registration Date': '', 'Mod

### 1E - Save the inspection evidence

The JSON stores header checks, the sampling method, observed values, and registry counts. The CSV stores one header-summary row per traffic file. These are inspection reports, not a cleaning audit or final research results. No source data is changed.

In [7]:
inspection_folder = ROOT / "reports" / "inspection"
inspection_folder.mkdir(parents=True, exist_ok=True)
inspection_summary = {
    "scope": "Source inspection only; no cleaning, typed ingestion, or Spark analysis performed",
    "traffic": {
        "file_count": len(traffic_files),
        "total_file_bytes": sum(path.stat().st_size for path in traffic_files),
        "header_summary": header_summary,
        "headers_by_file": traffic_headers,
        "sampling_method": "100 records after each byte position: 0%, 25%, 50%, 75%, 99%; not statistically representative",
        "window_summary": window_summary,
        "sample_row_count": len(sample_rows),
        "sample_label_counts": dict(sample_labels),
        "sample_protocol_counts": dict(sample_protocols),
        "sample_attempted_category_counts": dict(sample_attempted),
        "sample_timestamp_formats": dict(format_counts),
        "sample_timestamp_min": min(parsed_times).isoformat() if parsed_times else None,
        "sample_timestamp_max": max(parsed_times).isoformat() if parsed_times else None,
        "sample_unparsed_timestamp_count": len(unparsed_times),
        "sample_negative_sentinel_values": sentinel_examples,
        "raw_examples_by_observed_label": examples,
    },
    "iana": iana_summary,
}
summary_file = inspection_folder / "inspection_summary.json"
summary_file.write_text(json.dumps(inspection_summary, indent=2), encoding="utf-8")
with (inspection_folder / "traffic_headers.csv").open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(header_summary[0]))
    writer.writeheader()
    writer.writerows(header_summary)
print("Saved:", summary_file)
print("Saved:", inspection_folder / "traffic_headers.csv")

Saved: /home/jovyan/work/reports/inspection/inspection_summary.json
Saved: /home/jovyan/work/reports/inspection/traffic_headers.csv


**Step 1 complete.** Continue to Step 2 below to define and validate the explicit source schemas.


## Step 2 - Define the column types

A schema tells Spark the name and type of each CSV column. We define the types ourselves, as required by the assignment.

- `StringType`: text, identifiers, IP addresses and raw timestamps.
- `IntegerType`: ports, protocol codes and category codes.
- `LongType`: whole-number counts, byte totals and durations.
- `DoubleType`: decimal values, averages, rates and ratios.

`True` in `StructField` allows missing values so we can count and handle them later. We keep timestamps as text until the cleaning step. The two RST flag columns use doubles because some source values are written as `2.0`.

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType

TRAFFIC_SCHEMA = StructType([
    StructField("id", StringType(), True),
    StructField("Flow ID", StringType(), True),
    StructField("Src IP", StringType(), True),
    StructField("Src Port", IntegerType(), True),
    StructField("Dst IP", StringType(), True),
    StructField("Dst Port", IntegerType(), True),
    StructField("Protocol", IntegerType(), True),
    StructField("Timestamp", StringType(), True),
    StructField("Flow Duration", LongType(), True),
    StructField("Total Fwd Packet", LongType(), True),
    StructField("Total Bwd packets", LongType(), True),
    StructField("Total Length of Fwd Packet", LongType(), True),
    StructField("Total Length of Bwd Packet", LongType(), True),
    StructField("Fwd Packet Length Max", LongType(), True),
    StructField("Fwd Packet Length Min", LongType(), True),
    StructField("Fwd Packet Length Mean", DoubleType(), True),
    StructField("Fwd Packet Length Std", DoubleType(), True),
    StructField("Bwd Packet Length Max", LongType(), True),
    StructField("Bwd Packet Length Min", LongType(), True),
    StructField("Bwd Packet Length Mean", DoubleType(), True),
    StructField("Bwd Packet Length Std", DoubleType(), True),
    StructField("Flow Bytes/s", DoubleType(), True),
    StructField("Flow Packets/s", DoubleType(), True),
    StructField("Flow IAT Mean", DoubleType(), True),
    StructField("Flow IAT Std", DoubleType(), True),
    StructField("Flow IAT Max", LongType(), True),
    StructField("Flow IAT Min", LongType(), True),
    StructField("Fwd IAT Total", LongType(), True),
    StructField("Fwd IAT Mean", DoubleType(), True),
    StructField("Fwd IAT Std", DoubleType(), True),
    StructField("Fwd IAT Max", LongType(), True),
    StructField("Fwd IAT Min", LongType(), True),
    StructField("Bwd IAT Total", LongType(), True),
    StructField("Bwd IAT Mean", DoubleType(), True),
    StructField("Bwd IAT Std", DoubleType(), True),
    StructField("Bwd IAT Max", LongType(), True),
    StructField("Bwd IAT Min", LongType(), True),
    StructField("Fwd PSH Flags", LongType(), True),
    StructField("Bwd PSH Flags", LongType(), True),
    StructField("Fwd URG Flags", LongType(), True),
    StructField("Bwd URG Flags", LongType(), True),
    # These count-like source fields include decimal spellings such as "2.0".
    StructField("Fwd RST Flags", DoubleType(), True),
    StructField("Bwd RST Flags", DoubleType(), True),
    StructField("Fwd Header Length", LongType(), True),
    StructField("Bwd Header Length", LongType(), True),
    StructField("Fwd Packets/s", DoubleType(), True),
    StructField("Bwd Packets/s", DoubleType(), True),
    StructField("Packet Length Min", LongType(), True),
    StructField("Packet Length Max", LongType(), True),
    StructField("Packet Length Mean", DoubleType(), True),
    StructField("Packet Length Std", DoubleType(), True),
    StructField("Packet Length Variance", DoubleType(), True),
    StructField("FIN Flag Count", LongType(), True),
    StructField("SYN Flag Count", LongType(), True),
    StructField("RST Flag Count", LongType(), True),
    StructField("PSH Flag Count", LongType(), True),
    StructField("ACK Flag Count", LongType(), True),
    StructField("URG Flag Count", LongType(), True),
    StructField("CWR Flag Count", LongType(), True),
    StructField("ECE Flag Count", LongType(), True),
    StructField("Down/Up Ratio", DoubleType(), True),
    StructField("Average Packet Size", DoubleType(), True),
    StructField("Fwd Segment Size Avg", DoubleType(), True),
    StructField("Bwd Segment Size Avg", DoubleType(), True),
    StructField("Fwd Bytes/Bulk Avg", DoubleType(), True),
    StructField("Fwd Packet/Bulk Avg", DoubleType(), True),
    StructField("Fwd Bulk Rate Avg", DoubleType(), True),
    StructField("Bwd Bytes/Bulk Avg", DoubleType(), True),
    StructField("Bwd Packet/Bulk Avg", DoubleType(), True),
    StructField("Bwd Bulk Rate Avg", DoubleType(), True),
    StructField("Subflow Fwd Packets", LongType(), True),
    StructField("Subflow Fwd Bytes", LongType(), True),
    StructField("Subflow Bwd Packets", LongType(), True),
    StructField("Subflow Bwd Bytes", LongType(), True),
    StructField("FWD Init Win Bytes", LongType(), True),
    StructField("Bwd Init Win Bytes", LongType(), True),
    StructField("Fwd Act Data Pkts", LongType(), True),
    StructField("Fwd Seg Size Min", LongType(), True),
    StructField("Active Mean", DoubleType(), True),
    StructField("Active Std", DoubleType(), True),
    StructField("Active Max", LongType(), True),
    StructField("Active Min", LongType(), True),
    StructField("Idle Mean", DoubleType(), True),
    StructField("Idle Std", DoubleType(), True),
    StructField("Idle Max", LongType(), True),
    StructField("Idle Min", LongType(), True),
    StructField("ICMP Code", IntegerType(), True),
    StructField("ICMP Type", IntegerType(), True),
    StructField("Total TCP Flow Time", LongType(), True),
    StructField("Label", StringType(), True),
    StructField("Attempted Category", IntegerType(), True),
])

print("Traffic schema:", len(TRAFFIC_SCHEMA), "columns")

Traffic schema: 91 columns


### IANA schema

Keep IANA fields as text for now. Port values can contain ranges such as `225-241`, so we cannot read every port value as a single integer. Cleaning and conversion will come later.

In [9]:
IANA_SCHEMA = StructType([
    StructField("Service Name", StringType(), True),
    StructField("Port Number", StringType(), True),
    StructField("Transport Protocol", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Assignee", StringType(), True),
    StructField("Contact", StringType(), True),
    StructField("Registration Date", StringType(), True),
    StructField("Modification Date", StringType(), True),
    StructField("Reference", StringType(), True),
    StructField("Service Code", StringType(), True),
    StructField("Unauthorized Use Reported", StringType(), True),
    StructField("Assignment Notes", StringType(), True),
])

print("IANA schema:", len(IANA_SCHEMA), "columns")

IANA schema: 12 columns


### Check the column names and order

Compare the headers inspected in Step 1 with the schemas. An `assert` stops execution if a header does not match. This checks structure only; we will test reading values and writing Parquet in Step 3.

In [10]:
for file_name, header in traffic_headers.items():
    assert header == TRAFFIC_SCHEMA.fieldNames(), f"Check the columns in {file_name}"
    print("Header matches:", file_name)

assert registry_header == IANA_SCHEMA.fieldNames(), "Check the IANA columns"
print("IANA header matches.")

Header matches: Friday-02-03-2018.csv
Header matches: Friday-16-02-2018.csv
Header matches: Friday-23-02-2018.csv
Header matches: Thursday-01-03-2018.csv
Header matches: Thursday-15-02-2018.csv
Header matches: Thursday-22-02-2018.csv
Header matches: Tuesday-20-02-2018.csv
Header matches: Wednesday-14-02-2018.csv
Header matches: Wednesday-21-02-2018.csv
Header matches: Wednesday-28-02-2018.csv
IANA header matches.


**Step 2 complete.** Both schemas match the source headers. Continue to the small-sample ingestion test below.


## Step 3 - Test small-sample ingestion and Parquet

We will read CSV files with our explicit schemas, write typed Parquet, and read it back. Use only the 5,000 traffic records already inspected in Step 1, plus the small IANA registry. The original source files remain unchanged.

This step requires a working Spark/Java environment. Run the earlier cells first. The sample is for testing the pipeline, not measuring the full dataset's attack distribution.

### 3A - Save a small traffic CSV and start Spark

Save the inspected traffic records under `data/intermediate/`, so Spark can read a real CSV file without scanning the large originals. The sample file is replaced when this cell is rerun. Spark uses two local worker threads and four shuffle partitions.

In [11]:
sample_folder = ROOT / "data" / "intermediate"
sample_folder.mkdir(parents=True, exist_ok=True)
traffic_sample_file = sample_folder / "traffic_sample.csv"
with traffic_sample_file.open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=TRAFFIC_SCHEMA.fieldNames())
    writer.writeheader()
    writer.writerows(sample_rows)

from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]")
         .appName("DPA-Volume-Sample")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Sample traffic records:", len(sample_rows))

Sample traffic records: 5000


### 3B - Read both CSVs using the schemas

`header` reads column names; `enforceSchema=False` checks them against the schema. `FAILFAST` stops on parsing errors rather than silently skipping records. IANA also needs quoted fields and multiline descriptions handled correctly.

After reading, rename columns to lowercase with underscores for easier Spark SQL. This changes column names only. Blank source cells may be represented as null; missing-value treatment comes later.

In [12]:
traffic_df = (spark.read.schema(TRAFFIC_SCHEMA)
              .option("header", True).option("enforceSchema", False)
              .option("mode", "FAILFAST").option("escape", '"')
              .csv(str(traffic_sample_file)))

iana_df = (spark.read.schema(IANA_SCHEMA)
           .option("header", True).option("enforceSchema", False)
           .option("mode", "FAILFAST").option("escape", '"')
           .option("multiLine", True).csv(str(registry_file)))

traffic_df = traffic_df.toDF(*[name.lower().replace(" ", "_").replace("/", "_") for name in traffic_df.columns])
iana_df = iana_df.toDF(*[name.lower().replace(" ", "_") for name in iana_df.columns])

traffic_df.select("dst_port", "protocol", "flow_duration", "label").printSchema()
traffic_df.select("dst_port", "protocol", "flow_duration", "label").show(5)

root
 |-- dst_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- label: string (nullable = true)



+--------+--------+-------------+------+
|dst_port|protocol|flow_duration| label|
+--------+--------+-------------+------+
|      80|       6|    116659398|BENIGN|
|     443|       6|    117286103|BENIGN|
|     123|      17|     64032780|BENIGN|
|     443|       6|    118622137|BENIGN|
|     443|       6|       149794|BENIGN|
+--------+--------+-------------+------+
only showing top 5 rows


### 3C - Write Parquet, read it back, and check

Parquet stores column types. Rerunning this cell replaces only these test outputs under `data/parquet/sample/`.

We check the expected row counts and column types. `exceptAll` checks for changed or missing rows, including duplicate occurrences. Equal counts plus no missing rows confirm that the contents were preserved. No rows are cleaned or removed here.

In [13]:
parquet_folder = ROOT / "data" / "parquet" / "sample"
traffic_df.write.mode("overwrite").parquet(str(parquet_folder / "traffic"))
iana_df.write.mode("overwrite").parquet(str(parquet_folder / "iana"))

traffic_parquet = spark.read.parquet(str(parquet_folder / "traffic"))
iana_parquet = spark.read.parquet(str(parquet_folder / "iana"))

for name, original, restored, expected in [
    ("Traffic", traffic_df, traffic_parquet, len(sample_rows)),
    ("IANA", iana_df, iana_parquet, len(registry_rows)),
]:
    actual = restored.count()
    assert actual == expected, f"{name}: row count changed"
    assert original.dtypes == restored.dtypes, f"{name}: column types changed"
    assert original.exceptAll(restored).count() == 0, f"{name}: row contents changed"
    print(f"{name}: {actual} rows; types and contents preserved.")

print("Step 3 passed. Parquet saved in:", parquet_folder)

Traffic: 5000 rows; types and contents preserved.


IANA: 14536 rows; types and contents preserved.
Step 3 passed. Parquet saved in: /home/jovyan/work/data/parquet/sample


**Step 3 complete.** The traffic sample and IANA registry have been read with explicit schemas and saved as typed Parquet. Counts, types and contents were checked after reading them back.

Next is Step 4: inspect data quality before deciding cleaning rules. Full traffic ingestion and research analysis are still pending.